# Treino de um Laya dedicado — Studio Lab (GPU T4)

Wrapper fino: todo o trabalho é feito pela CLI `laya-pipeline`; este notebook só prepara o ambiente e chama os comandos, na ordem.

**Antes:** na estação, a tarefa já foi rotulada, dividida, revisada (`approve data`) e enviada com `laya-pipeline sync push`.

**Sessão caiu?** Rode as células de novo: o treino retoma da última época salva (`runs/<versão>/train_state.pt`).

In [ ]:
# 1. GPU visível?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Repositório da tarefa (troque pela URL do repositório criado a partir do laya-template)
TASK_REPO = "https://github.com/laredonunes/laya-EXEMPLO.git"
TASK_DIR = "/home/studio-lab-user/" + TASK_REPO.rstrip("/").split("/")[-1].removesuffix(".git")

import os, subprocess
if os.path.isdir(TASK_DIR):
    subprocess.run(["git", "-C", TASK_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", TASK_REPO, TASK_DIR], check=True)
os.chdir(TASK_DIR)
print(open("requirements.txt").read())

In [ ]:
# 3. Instala o pipeline na versão fixada pelo repositório da tarefa
!pip install -q -r requirements.txt

In [ ]:
# 4. Acesso ao proxy (dados e publicação passam por ele). O token não fica salvo no notebook.
import getpass
os.environ["HERMES_PROXY_URL"] = input("HERMES_PROXY_URL: ").strip()
os.environ["HERMES_TOKEN"] = getpass.getpass("HERMES_TOKEN: ")

In [ ]:
# 5. Baixa os dados rotulados e as aprovações; mostra o estado
!laya-pipeline sync pull
!laya-pipeline status

In [ ]:
# 6. Baseline (modelo base com o contexto da tarefa), se ainda não existir
!test -f runs/*/eval_baseline.json || laya-pipeline eval

In [ ]:
# 7. Treino (retoma sozinho se já havia começado)
!laya-pipeline train

In [ ]:
# 8. Avalia o treinado e devolve os relatórios para a estação
!laya-pipeline eval --trained
!laya-pipeline sync push
!cat runs/*/report.md

## Publicação

Só depois de revisar o `report.md` e aprovar (`laya-pipeline approve report --by <nome>`, aqui ou na estação + `sync push`).

O checkpoint vai do notebook direto para o S3 (URL pré-assinada), então não precisa trazê-lo para a estação.

In [ ]:
# 9. Publica checkpoint + relatórios em models/laya/<tarefa>/<versão>/
!laya-pipeline sync pull
!laya-pipeline publish